# Video Deep Learning Features — SwinT / FabNet / CLIP / DINOv2

| Model | Wrapper | Output dim | Input size |
|---|---|---|---|
| Swin Transformer | `SwinTWrapper` | 768 | 224×224 |
| FAb-Net | `FabNetWrapper` | 256 | 112×112 |
| CLIP ViT-H/14 | `ClipWrapper` | 1024 | 224×224 |
| DINOv2 (base) | `DINOv2Wrapper` | 768 | 224×224 |

Each wrapper handles preprocessing internally — pass a raw `(T, 3, H, W)` uint8 tensor and get a `(T, D)` float feature tensor back.
Demo uses `multispeaker_short.mp4` sampled at 5 fps.

In [1]:
from pathlib import Path

from exordium.video.core.io import load_video

video_path = Path("../tests/fixtures/video/multispeaker_short.mp4")

# Load frames as uint8 tensor (T, 3, H, W)
frames, fps = load_video(video_path, fps=5)
print(f"Loaded {frames.shape[0]} frames @ {fps} fps")
print(f"Shape : {tuple(frames.shape)}  dtype={frames.dtype}")

objc[43513]: Class AVFFrameReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x118cf03a8) and /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x13b5703a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[43513]: Class AVFAudioReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x118cf03f8) and /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x13b5703f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
/Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets

Loaded 5 frames @ 5 fps
Shape : (5, 3, 720, 1280)  dtype=torch.uint8


## 1. Swin Transformer

In [2]:
from exordium.video.deep import SwinTWrapper

swin = SwinTWrapper(device_id=-1)  # -1 → CPU
swin_feats = swin(frames)  # (T, 768) Tensor

print(f"Output: {tuple(swin_feats.shape)}  dtype={swin_feats.dtype}")

Output: (5, 768)  dtype=torch.float32


## 2. FAb-Net

Pre-trained on face sequences for identity-invariant appearance features.
Expects face crops; here we feed full frames for illustration.

In [3]:
from exordium.video.deep import FabNetWrapper

fabnet = FabNetWrapper(device_id=-1)
fabnet_feats = fabnet(frames)  # (T, 256) Tensor

print(f"Output: {tuple(fabnet_feats.shape)}  dtype={fabnet_feats.dtype}")

2026-03-26 14:34:28 INFO FAb-Net is loaded to cpu.


Output: (5, 256)  dtype=torch.float32


## 3. CLIP

In [4]:
from exordium.video.deep import ClipWrapper

clip = ClipWrapper(device_id=-1)
clip_feats = clip(frames)  # (T, 1024) Tensor

print(f"Output: {tuple(clip_feats.shape)}  dtype={clip_feats.dtype}")

2026-03-26 14:34:28 INFO HTTP Request: HEAD https://huggingface.co/laion/CLIP-ViT-H-14-laion2B-s32B-b79K/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 14:34:28 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/laion/CLIP-ViT-H-14-laion2B-s32B-b79K/1c2b8495b28150b8a4922ee1c8edee224c284c0c/config.json "HTTP/1.1 200 OK"
2026-03-26 14:34:29 INFO HTTP Request: HEAD https://huggingface.co/laion/CLIP-ViT-H-14-laion2B-s32B-b79K/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 520/520 [00:00<00:00, 25820.57it/s]
CLIPVisionModelWithProjection LOAD REPORT from: laion/CLIP-ViT-H-14-laion2B-s32B-b79K
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
te

Output: (5, 1024)  dtype=torch.float32


## 5. Feature Summary

## 4. DINOv2

Self-supervised ViT trained without language supervision (Facebook Research).
Four variants available: `small` (384-d), `base` (768-d), `large` (1024-d), `giant` (1536-d).
Outputs L2-normalised CLS-token embeddings.

In [5]:
from exordium.video.deep.dinov2 import DINOv2Wrapper

dino = DINOv2Wrapper(model_name="base", device_id=-1)  # -1 → CPU
dino_feats = dino(frames)  # (T, 768) L2-normalised CLS-token embeddings

print(f"Output: {tuple(dino_feats.shape)}  dtype={dino_feats.dtype}")
print(f"L2 norms (should be ~1.0): {dino_feats.norm(dim=-1).tolist()}")

2026-03-26 14:34:31 INFO HTTP Request: HEAD https://huggingface.co/facebook/dinov2-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 14:34:31 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/dinov2-base/f9e44c814b77203eaa57a6bdbbd535f21ede1415/config.json "HTTP/1.1 200 OK"
2026-03-26 14:34:31 INFO HTTP Request: HEAD https://huggingface.co/facebook/dinov2-base/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████| 223/223 [00:00<00:00, 10695.96it/s]


Output: (5, 768)  dtype=torch.float32
L2 norms (should be ~1.0): [1.0, 1.0000001192092896, 0.9999999403953552, 1.0, 1.0]


In [6]:
print(f"Frames: {frames.shape[0]}  ({fps} fps)")
print()
print(f"{'Model':<22} {'Output shape':<20} {'Dim':>6}")
print("-" * 50)
for name, feats in [
    ("Swin Transformer", swin_feats),
    ("FabNet", fabnet_feats),
    ("CLIP ViT-H/14", clip_feats),
    ("DINOv2 base", dino_feats),
]:
    print(f"{name:<22} {str(tuple(feats.shape)):<20} {feats.shape[-1]:>6}")

Frames: 5  (5 fps)

Model                  Output shape            Dim
--------------------------------------------------
Swin Transformer       (5, 768)                768
FabNet                 (5, 256)                256
CLIP ViT-H/14          (5, 1024)              1024
DINOv2 base            (5, 768)                768
